In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 06b_save_transformation_artifacts
# MAGIC 
# MAGIC **Objetivo**: Guardar artefactos del pipeline de transformación para inferencia
# MAGIC 
# MAGIC **Artefactos a guardar**:
# MAGIC - scaler.pkl (StandardScaler entrenado)
# MAGIC - pca.pkl (PCA entrenado)
# MAGIC - features_retained.csv (lista de features seleccionadas)
# MAGIC - transformation_metadata.csv (metadata del pipeline)
# MAGIC 
# MAGIC **Nota**: Este código debe agregarse AL FINAL del notebook 06_feature_correlation_pca_reduction

# COMMAND ----------

# MAGIC %md
# MAGIC ## ⚠️ IMPORTANTE: Agregar al final del notebook 06
# MAGIC 
# MAGIC Este código debe ejecutarse DESPUÉS de:
# MAGIC - Eliminar features por correlación
# MAGIC - Entrenar StandardScaler
# MAGIC - Entrenar PCA
# MAGIC 
# MAGIC Las variables necesarias son:
# MAGIC - `scaler`: StandardScaler entrenado
# MAGIC - `pca`: PCA entrenado
# MAGIC - `features_keep` o `features_retained`: lista de features seleccionadas
# MAGIC - `feature_cols`: lista original de features
# MAGIC - `n_comp` o `n_components`: número de componentes PCA

# COMMAND ----------

import joblib
import pandas as pd
from datetime import datetime

MODELS_PATH = "/Volumes/olist/olist_gold/models/"

print("=" * 80)
print("💾 GUARDANDO ARTEFACTOS PARA INFERENCIA")
print("=" * 80)
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Crear volume models

# COMMAND ----------

print("🔧 Verificando estructura...")
print()

try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.models")
    print("✅ Volume 'models' verificado/creado")
except Exception as e:
    print(f"⚠️  Volume models: {e}")

print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Guardar StandardScaler

# COMMAND ----------

print("💾 Guardando StandardScaler...")
print()

try:
    # Guardar en DBFS
    scaler_path = f"/dbfs{MODELS_PATH}scaler.pkl"
    joblib.dump(scaler, scaler_path)
    
    print(f"✅ Scaler guardado en: {scaler_path}")
    print(f"   • Features: {scaler.n_features_in_}")
    print(f"   • Mean shape: {scaler.mean_.shape}")
    print(f"   • Scale shape: {scaler.scale_.shape}")
    
except Exception as e:
    print(f"❌ Error guardando scaler: {e}")
    raise

print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Guardar PCA

# COMMAND ----------

print("💾 Guardando PCA...")
print()

try:
    # Guardar en DBFS
    pca_path = f"/dbfs{MODELS_PATH}pca.pkl"
    joblib.dump(pca, pca_path)
    
    print(f"✅ PCA guardado en: {pca_path}")
    print(f"   • Componentes: {pca.n_components_}")
    print(f"   • Features entrada: {pca.n_features_in_}")
    print(f"   • Varianza explicada: {pca.explained_variance_ratio_.sum()*100:.2f}%")
    
except Exception as e:
    print(f"❌ Error guardando PCA: {e}")
    raise

print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Guardar lista de features retenidas

# COMMAND ----------

print("💾 Guardando lista de features retenidas...")
print()

try:
    # Verificar qué variable existe (puede variar según el notebook)
    if 'features_keep' in locals() or 'features_keep' in globals():
        features_list = features_keep
    elif 'features_retained' in locals() or 'features_retained' in globals():
        features_list = features_retained
    else:
        raise NameError("No se encontró 'features_keep' ni 'features_retained'")
    
    # Crear DataFrame
    features_df = pd.DataFrame({
        'feature': features_list,
        'order': range(len(features_list))
    })
    
    # Guardar usando método temporal para CSV
    temp_path = f"{MODELS_PATH}features_temp/"
    spark.createDataFrame(features_df).write \
        .format("csv").mode("overwrite").option("header", "true") \
        .save(temp_path)
    
    # Mover el archivo CSV
    csv_files = [f for f in dbutils.fs.ls(temp_path) if f.name.endswith('.csv')]
    if csv_files:
        dbutils.fs.cp(csv_files[0].path, f"{MODELS_PATH}features_retained.csv")
    
    dbutils.fs.rm(temp_path, True)
    
    print(f"✅ Features retenidas guardadas: {MODELS_PATH}features_retained.csv")
    print(f"   • Total features: {len(features_list)}")
    
except Exception as e:
    print(f"❌ Error guardando features retenidas: {e}")
    raise

print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Guardar metadata del pipeline

# COMMAND ----------

print("💾 Guardando metadata del pipeline...")
print()

try:
    # Determinar número de componentes PCA
    if 'n_comp' in locals() or 'n_comp' in globals():
        n_components_used = n_comp
    elif 'n_components' in locals() or 'n_components' in globals():
        n_components_used = n_components
    else:
        n_components_used = pca.n_components_
    
    # Crear metadata
    metadata = pd.DataFrame([{
        'n_features_original': len(feature_cols) if 'feature_cols' in locals() or 'feature_cols' in globals() else len(features_list) + len(to_drop),
        'n_features_after_correlation': len(features_list),
        'n_features_eliminated': len(to_drop) if 'to_drop' in locals() or 'to_drop' in globals() else 0,
        'n_pca_components': n_components_used,
        'pca_variance_explained': pca.explained_variance_ratio_.sum(),
        'correlation_threshold': 0.85,
        'training_date': datetime.now().strftime('%Y-%m-%d'),
        'cutoff_date': '2018-09-30 23:59:59',
        'scaler_type': 'StandardScaler',
        'pca_type': 'PCA'
    }])
    
    # Guardar usando método temporal
    temp_path = f"{MODELS_PATH}metadata_temp/"
    spark.createDataFrame(metadata).write \
        .format("csv").mode("overwrite").option("header", "true") \
        .save(temp_path)
    
    # Mover el archivo CSV
    csv_files = [f for f in dbutils.fs.ls(temp_path) if f.name.endswith('.csv')]
    if csv_files:
        dbutils.fs.cp(csv_files[0].path, f"{MODELS_PATH}transformation_metadata.csv")
    
    dbutils.fs.rm(temp_path, True)
    
    print(f"✅ Metadata guardada: {MODELS_PATH}transformation_metadata.csv")
    print()
    print("Contenido:")
    for col in metadata.columns:
        print(f"   • {col}: {metadata[col].iloc[0]}")
    
except Exception as e:
    print(f"❌ Error guardando metadata: {e}")
    raise

print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Verificación final

# COMMAND ----------

print("=" * 80)
print("🔍 VERIFICACIÓN FINAL DE ARTEFACTOS")
print("=" * 80)
print()

artifacts_to_check = {
    "scaler.pkl": "StandardScaler",
    "pca.pkl": "PCA",
    "features_retained.csv": "Features retenidas",
    "transformation_metadata.csv": "Metadata"
}

all_ok = True

for artifact, description in artifacts_to_check.items():
    path = f"{MODELS_PATH}{artifact}"
    try:
        files = dbutils.fs.ls(path)
        print(f"✅ {artifact:35s} - {description}")
    except:
        print(f"❌ {artifact:35s} - NO ENCONTRADO")
        all_ok = False

print()

if all_ok:
    print("✅ TODOS LOS ARTEFACTOS GUARDADOS EXITOSAMENTE")
    print()
    print("🎯 SIGUIENTE PASO:")
    print("   Los artefactos están listos para el pipeline de inferencia")
    print("   Ejecutar: 10_inference_pipeline_production")
else:
    print("❌ ALGUNOS ARTEFACTOS NO SE GUARDARON CORRECTAMENTE")
    print("   Revisa los errores arriba")

print()
print("=" * 80)